# Notebook optimizado en memoria

**Cambios principales respecto de la versión original** (detalle en cada celda marcada con `# [MEM]`):

1. El loop principal ahora llama a una **función `process_halo()`**: todas las variables de un halo son locales y se liberan solas al terminar. Se elimina el truco de `del globals()[...]`, que no liberaba la memoria que realmente pesa.
2. **KD-tree por snapshot** para el gas del halo: antes, para *cada* subhalo se calculaban ~10 arrays temporales del tamaño de **todas** las partículas de gas del cúmulo (millones). Ahora sólo se tocan las partículas cercanas al subhalo. Mismo resultado, mucha menos RAM pico y mucho más rápido.
3. Se cargan **sólo los campos que se usan** (`Density` no se usaba en ningún lado).
4. Se borran `data_halos` y `data_subhalos` (catálogos completos de TNG300, ~1–2 GB) apenas se dejan de usar.
5. Los `print` por subhalo van a un **archivo de log** (`log_download.txt`): la salida de la celda original tenía cientos de miles de líneas que quedan en la RAM del servidor de Jupyter y del navegador.
6. Backend `Agg` de matplotlib + `plt.close('all')` por snapshot, para que ninguna figura quede viva.
7. El HDF5 se escribe al final de cada halo y se hace `flush()`. Si el kernel muere a mitad de un halo, no queda un grupo a medio escribir que después se saltee.
8. `pd.concat` / `np.vstack` dentro de loops (copias cuadráticas) reemplazados por listas + un solo concat.

In [1]:
import h5py
import numpy as np
import matplotlib
matplotlib.use('Agg')  # [MEM] sin backend inline: las figuras no se acumulan en el notebook
import matplotlib.pyplot as plt
from illustris_python.groupcat import loadSingle, loadHeader
import illustris_python as il
import requests
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import tempfile
import contextlib
import os
from datetime import datetime
import time
import gc
import ctypes
from scipy.spatial import cKDTree

In [2]:
# Constante de Hubble en unidades de [km/s / Mpc]:
H0 = 67.74
h = H0 / 100

# Tamaño del box de simulación
box_size = 204.98124  # Mpc/h

# Path de las simulaciones TNG300-1 en este directorio:
myBasePath = '../sims.TNG/TNG300-1/output'

K_B = 1.38 * 1e-16  # erg/K

LOG_FILE = 'log_download.txt'

In [3]:
# [MEM] Utilidades de memoria y log

def mem_gb():
    '''RAM residente actual del proceso (GB). Solo Linux.'''
    try:
        with open('/proc/self/status') as f:
            for line in f:
                if line.startswith('VmRSS:'):
                    return int(line.split()[1]) / 1024**2
    except Exception:
        pass
    return float('nan')

def liberar_memoria():
    '''Fuerza al recolector y le devuelve al sistema operativo la memoria libre de glibc.'''
    gc.collect()
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass

def log_msg(msg, to_screen=False):
    '''Escribe en el archivo de log; solo imprime en pantalla si to_screen=True.'''
    with open(LOG_FILE, 'a') as f:
        f.write(msg + '\n')
    if to_screen:
        print(msg)

# PASO 1: BAJAR HALOS QUE A Z=0 TENGAN $M_{200}$ EN [Msun/h] MAYOR A 13.5

In [4]:
# Traemos algunas propiedades de TODOS los halos de las TNG300
data_halos = il.groupcat.loadHalos(myBasePath, 99, fields=['GroupFirstSub', 'Group_M_Crit200', 'Group_R_Crit200', 'GroupPos', 'GroupVel'])

In [5]:
# [MEM] Seleccionamos ANTES de armar el DataFrame (antes se armaba un DataFrame de ~11 millones de filas)
with np.errstate(divide='ignore'):
    m200_log_all = np.log10(data_halos['Group_M_Crit200'] * 1e10)  # Msun/h

sel = np.where(m200_log_all >= 13.5)[0]

halos = pd.DataFrame({
    'id_z0': sel,
    'id_central_galaxy_z0': data_halos['GroupFirstSub'][sel],
    'm200_z0': m200_log_all[sel],                          # Msun/h
    'r200_z0': data_halos['Group_R_Crit200'][sel] / 1000,  # cMpc/h
    'pos_x_z0': data_halos['GroupPos'][sel, 0] / 1000,     # cMpc/h
    'pos_y_z0': data_halos['GroupPos'][sel, 1] / 1000,     # cMpc/h
    'pos_z_z0': data_halos['GroupPos'][sel, 2] / 1000,     # cMpc/h
    'vel_x_z0': data_halos['GroupVel'][sel, 0],            # km/s
    'vel_y_z0': data_halos['GroupVel'][sel, 1],            # km/s
    'vel_z_z0': data_halos['GroupVel'][sel, 2]             # km/s
})
len(halos)

754

# PASO 2: CRITERIO DE AISLAMIENTO

In [6]:
def periodic_distance(pos1, pos2, box_size):
    """
    Calcula la distancia periódica entre una posición y N posiciones.
    pos1 : array (3,) ; pos2 : array (N, 3) ; box_size : float
    """
    delta = np.abs(pos2 - pos1)
    delta = np.minimum(delta, box_size - delta)
    return np.sqrt(np.sum(delta**2, axis=1))

In [7]:
def filter_isolated_halos(halos_df, m200_log_all, all_positions, box_size, verbose=True):
    '''
    Descarta un halo si tiene otro halo con masa > 10% de su m200 dentro de 5*r200
    (con corrección de periodicidad).

    [MEM] Solo se guardan los halos que pueden llegar a ser "vecinos masivos"
    (log M > min(m200) - 1), unas decenas de miles en vez de 11 millones.
    Además se conservan los IDs originales: antes se filtraba por masa > 0 y después
    se comparaba el índice del array filtrado con halo_id, lo que solo era correcto
    mientras ningún halo con masa 0 apareciera antes que los halos de interés.
    '''
    min_log = halos_df['m200_z0'].min() + np.log10(0.1)
    cand_ids = np.where(m200_log_all > min_log)[0]
    cand_mass = m200_log_all[cand_ids]
    cand_pos = all_positions[cand_ids] / 1000  # Mpc/h

    if verbose:
        print(f'Halos candidatos a vecino masivo: {len(cand_ids)}')
        print(f'Halos a verificar: {len(halos_df)}')

    isolated_indices = []
    for idx, halo in halos_df.iterrows():
        halo_id = halo['id_z0']
        min_mass_log = halo['m200_z0'] + np.log10(0.1)
        search_radius = 5 * halo['r200_z0']
        halo_pos = np.array([halo['pos_x_z0'], halo['pos_y_z0'], halo['pos_z_z0']])

        massive_mask = (cand_mass > min_mass_log) & (cand_ids != halo_id)
        if np.any(massive_mask):
            distances = periodic_distance(halo_pos, cand_pos[massive_mask], box_size)
            if not np.any(distances < search_radius):
                isolated_indices.append(idx)
        else:
            isolated_indices.append(idx)

    isolated_halos = halos_df.loc[isolated_indices].copy()

    if verbose:
        print(f'Halos aislados: {len(isolated_halos)}')
        print(f'Halos descartados: {len(halos_df) - len(isolated_halos)}')
        print(f'Fracción aislada: {len(isolated_halos)/len(halos_df):.1%}')

    return isolated_halos

In [8]:
isolated_halos = filter_isolated_halos(halos, m200_log_all, data_halos['GroupPos'], box_size)

# [MEM] El catálogo completo de halos ya no se usa más
del data_halos, m200_log_all, sel
liberar_memoria()
print(f'RAM: {mem_gb():.2f} GB')

Halos candidatos a vecino masivo: 7900
Halos a verificar: 754
Halos aislados: 549
Halos descartados: 205
Fracción aislada: 72.8%
RAM: 0.19 GB


# PASO 3: TRAEMOS TODAS LAS GALAXIAS CON $M_{\star}$ EN [Msun/h] MAYOR A 9.

In [9]:
data_subhalos = il.groupcat.loadSubhalos(myBasePath, 99, fields=['SubhaloFlag', 'SubhaloMassType', 'SubhaloPos', 'SubhaloVel'])

In [10]:
# [MEM] Filtramos antes de armar el DataFrame y después borramos el catálogo completo
with np.errstate(divide='ignore'):
    mstar_log_all = np.log10(data_subhalos['SubhaloMassType'][:, 4] * 1e10)  # Msun/h

sel = np.where((mstar_log_all >= 9.) & (data_subhalos['SubhaloFlag'] != 0))[0]

subhalos = pd.DataFrame({
    'sub_id_z0': sel,
    'sub_mstar_z0': mstar_log_all[sel],                     # Msun/h
    'sub_pos_x_z0': data_subhalos['SubhaloPos'][sel, 0] / 1000,  # cMpc/h
    'sub_pos_y_z0': data_subhalos['SubhaloPos'][sel, 1] / 1000,  # cMpc/h
    'sub_pos_z_z0': data_subhalos['SubhaloPos'][sel, 2] / 1000,  # cMpc/h
    'sub_vel_x_z0': data_subhalos['SubhaloVel'][sel, 0],    # km/s
    'sub_vel_y_z0': data_subhalos['SubhaloVel'][sel, 1],    # km/s
    'sub_vel_z_z0': data_subhalos['SubhaloVel'][sel, 2],    # km/s
    'flag': data_subhalos['SubhaloFlag'][sel]
})

del data_subhalos, mstar_log_all, sel
liberar_memoria()
print(f'RAM: {mem_gb():.2f} GB')
len(subhalos)

RAM: 0.20 GB


237222

In [11]:
# Calculamos, para cada halo, la distancia 3D (periódica) a todas las galaxias.
# Si es menor a 5*r200 la agregamos al DataFrame 'data'.
# [MEM] Se juntan los pedazos en una lista y se hace un solo concat al final
# (antes pd.concat dentro del loop copiaba todo 'data' en cada iteración).

sub_pos_all = subhalos[['sub_pos_x_z0', 'sub_pos_y_z0', 'sub_pos_z_z0']].to_numpy()
piezas = []

for _, halo in tqdm(isolated_halos.iterrows(), total=len(isolated_halos)):
    halo_pos = np.array([halo['pos_x_z0'], halo['pos_y_z0'], halo['pos_z_z0']])  # cMpc/h
    dist = periodic_distance(halo_pos, sub_pos_all, box_size)                      # cMpc/h
    in_range = dist <= 5 * halo['r200_z0']

    if np.any(in_range):
        pieza = subhalos[in_range].copy()
        pieza['sub_dist_3d_z0'] = dist[in_range]
        pieza.insert(0, 'halo_id_z0', halo['id_z0'])
        pieza['halo_id_central_galaxy_z0'] = halo['id_central_galaxy_z0']
        pieza['halo_r200_z0'] = halo['r200_z0']
        pieza['halo_m200_z0'] = halo['m200_z0']
        pieza['halo_pos_x_z0'] = halo['pos_x_z0']
        pieza['halo_pos_y_z0'] = halo['pos_y_z0']
        pieza['halo_pos_z_z0'] = halo['pos_z_z0']
        pieza['halo_vel_x_z0'] = halo['vel_x_z0']
        pieza['halo_vel_y_z0'] = halo['vel_y_z0']
        pieza['halo_vel_z_z0'] = halo['vel_z_z0']
        piezas.append(pieza)

data = pd.concat(piezas, ignore_index=True)
del piezas, sub_pos_all
liberar_memoria()
len(data)

100%|██████████| 549/549 [00:05<00:00, 93.75it/s]


39132

In [12]:
data['halo_id_z0'][data['sub_id_z0'] == 1]

1        0.0
7306    30.0
Name: halo_id_z0, dtype: float64

# PASO 4: CORREGIMOS LAS POSICIONES DE LAS GALAXIAS POR PERIODICIDAD

In [13]:
# [MEM] Vectorizado (antes: iterrows + data.loc fila por fila)
corrections = np.full(len(data), '', dtype=object)

for eje in ['x', 'y', 'z']:
    col = f'sub_pos_{eje}_z0'
    d_raw = data[col].to_numpy() - data[f'halo_pos_{eje}_z0'].to_numpy()
    # d_raw > +box/2: subhalo muy adelante -> restamos box ; d_raw < -box/2: muy atrás -> sumamos box
    mas = d_raw > box_size / 2
    menos = d_raw < -box_size / 2
    data.loc[mas, col] -= box_size
    data.loc[menos, col] += box_size
    corregido = mas | menos
    corrections[corregido] = corrections[corregido] + eje.upper()

data['pos_corr'] = pd.Series(corrections).replace('', np.nan)

print(f"Subhalos con correcciones: {data['pos_corr'].notna().sum()}")
print(f"\nDistribución de correcciones:")
print(data['pos_corr'].value_counts().sort_index())

Subhalos con correcciones: 152

Distribución de correcciones:
pos_corr
X    21
Y    51
Z    80
Name: count, dtype: int64


In [14]:
data = data.drop(['flag', 'pos_corr'], axis=1)

In [15]:
data['sub_id_z0'] == 2

0        False
1        False
2         True
3        False
4        False
         ...  
39127    False
39128    False
39129    False
39130    False
39131    False
Name: sub_id_z0, Length: 39132, dtype: bool

# PASO 4: MERGER TREE DE LOS SUBHALOS

In [16]:
def apply_periodic_correction(sub_pos, halo_pos, box_size):
    """
    Aplica corrección de periodicidad a la posición del subhalo respecto al halo.
    """
    corrected_pos = sub_pos.copy()
    for k in range(3):
        d = sub_pos[k] - halo_pos[k]
        if d > box_size/2:
            corrected_pos[k] -= box_size
        elif d < -box_size/2:
            corrected_pos[k] += box_size
    return corrected_pos

In [17]:
def load_halo_evolution(myBasePath, halo_id_z0, id_central_galaxy_z0):
    """
    Carga la evolución del halo desde z=0 hasta el snapshot más antiguo disponible.

    Returns: dict con 'pos_x', 'pos_y', 'pos_z', 'r200', 'm200', 'snapshots', 'vel', 'haloID'
    """
    try:
        central_galaxy_info = il.sublink.loadTree(
            myBasePath, 99, id_central_galaxy_z0,
            fields=['SubhaloGrNr', 'SubhaloPos', 'SnapNum', 'SubhaloVel'],
            onlyMPB=True
        )
        halo_pos_x = central_galaxy_info['SubhaloPos'][:, 0] / 1000  # cMpc/h
        halo_pos_y = central_galaxy_info['SubhaloPos'][:, 1] / 1000  # cMpc/h
        halo_pos_z = central_galaxy_info['SubhaloPos'][:, 2] / 1000  # cMpc/h
        snapshots = central_galaxy_info.get('SnapNum', np.arange(99, 99-len(halo_pos_x), -1))
        halo_vel = central_galaxy_info['SubhaloVel']  # km/s (vel. de la central = vel. del halo)
    except Exception:
        return None

    halo_r200, halo_m200, halo_id = [], [], []
    for snap in snapshots:
        try:
            aux = central_galaxy_info['SubhaloGrNr'][np.where(central_galaxy_info['SnapNum'] == snap)[0][0]]
            halo_id.append(aux)
            halo_info = il.groupcat.loadSingle(myBasePath, snap, haloID=aux)
            halo_r200.append(round(halo_info['Group_R_Crit200'] / 1000, 5))           # cMpc/h
            halo_m200.append(round(np.log10(halo_info['Group_M_Crit200'] * 1e10), 5))  # Msun/h
        except Exception:
            halo_r200.append(np.nan)
            halo_m200.append(np.nan)
            halo_id.append(np.nan)

    return {
        'pos_x': np.array(halo_pos_x),
        'pos_y': np.array(halo_pos_y),
        'pos_z': np.array(halo_pos_z),
        'r200': np.array(halo_r200),
        'm200': np.array(halo_m200),
        'snapshots': np.array(snapshots),
        'vel': np.array(halo_vel),
        'haloID': halo_id
    }

In [18]:
def get(path, params=None, folderName=''):
    """
    Illustris function
    """
    r = requests.get(path, params=params, headers=headers)
    r.raise_for_status()

    if r.headers['content-type'] == 'application/json':
        return r.json()

    if 'content-disposition' in r.headers:
        filename = r.headers['content-disposition'].split("filename=")[1]
        if filename.endswith('.hdf5'):
            file_access_property_list = h5py.h5p.create(h5py.h5p.FILE_ACCESS)
            file_access_property_list.set_fapl_core(backing_store=False)
            file_access_property_list.set_file_image(r.content)

            file_id_args = {
                'fapl': file_access_property_list,
                'flags': h5py.h5f.ACC_RDONLY,
                'name': next(tempfile._get_candidate_names()).encode()
            }

            h5_file_args = {'backing_store': False, 'driver': 'core', 'mode': 'r'}
            with contextlib.closing(h5py.h5f.open(**file_id_args)) as file_id:
                with h5py.File(file_id, **h5_file_args) as h5_file:
                    if 'grid' in h5_file.keys(): return np.array(h5_file['grid'])
                    else:
                        results = []
                        for k in h5_file.keys():
                            for sk in h5_file[k].keys():
                                results.append(np.array(h5_file[k][sk]))
                        return results
        else:
            with open(folderName + filename, 'wb') as f:
                f.write(r.content)
            return filename
    return r

def get_url(which, subhalo_id, snap):
    return 'http://www.tng-project.org/api/TNG300-1/snapshots/' + str(snap) + '/' + which + '/' + str(subhalo_id)

headers = {"api-key": '81b7c70637fa8b110e6b9f236ea07c37'}

## Funciones del loop principal

[MEM] Todo lo que antes vivía como variable global dentro del loop ahora vive dentro de `process_halo()`. Al terminar cada halo, Python libera todas sus variables locales automáticamente.

In [19]:
# [MEM] Solo los campos que se usan (Density no se usaba)
HALO_GAS_FIELDS = ['Coordinates', 'ElectronAbundance', 'InternalEnergy', 'Masses']
SUB_GAS_FIELDS = ['Coordinates', 'Velocities', 'ElectronAbundance', 'InternalEnergy', 'Masses', 'Potential']
SUB_GAS_FIELDS_SIN_POT = ['Coordinates', 'Velocities', 'ElectronAbundance', 'InternalEnergy', 'Masses']

TREE_FIELDS = ['SubhaloGrNr', 'SubhaloPos', 'SnapNum', 'SubhaloHalfmassRad', 'SubhaloIDRaw',
               'SubhaloSFR', 'SubhaloStellarPhotometrics', 'SubhaloMass', 'SubhaloMassType', 'SubhaloVel']


def gas_stats(coords, masses, int_energy, elec_ab, sub_pos, halo_pos, hmr, vel_dir=None):
    """
    Estadísticas del gas alrededor de un subhalo, por regiones:
      [0] r < 50 kpc/h
      [1] r < 2 hmr
      [2] 2 hmr < r < 3 hmr del lado que mira al cúmulo
      [3] (solo si se pasa vel_dir) 2 hmr < r < 3 hmr del lado hacia donde apunta
          la velocidad del subhalo:  (r_part - r_sub) · v > 0
    coords en cMpc/h. Mismas fórmulas que la versión original.
    """
    d = coords - sub_pos
    dist = np.sqrt(np.einsum('ij,ij->i', d, d))  # Mpc/h
    ang = d @ (halo_pos - sub_pos)
    en_cascara = (dist > 2*hmr) & (dist < 3*hmr)

    sels = [np.where(dist < (50/1000))[0],
            np.where(dist < 2*hmr)[0],
            np.where(en_cascara & (ang > 0))[0]]
    if vel_dir is not None:
        sels.append(np.where(en_cascara & ((d @ vel_dir) > 0))[0])
    del d

    out = {k: np.zeros(len(sels)) for k in ('num', 'mass', 'intEnergy', 'elecAbundance', 'T')}
    for k, s in enumerate(sels):
        m, u, xe = masses[s], int_energy[s], elec_ab[s]
        mu = (4 * m) / (1 + 3*0.76 + 4*0.76*xe)  # auxiliar mu estimation
        out['num'][k] = len(s)
        out['mass'][k] = np.sum(m)
        out['intEnergy'][k] = np.sum(u)
        out['elecAbundance'][k] = np.sum(xe)
        out['T'][k] = np.sum((5/3 - 1) * u * mu / K_B)
    out['ind_fix'] = sels[0]
    out['ind_vel'] = sels[3] if vel_dir is not None else None
    return out


def cargar_subhalos_list(halo_subhalos, nsubhalos_max):
    """Merger trees (MPB) de los subhalos del halo, apilados en un array (N_filas, 27)."""
    filas = []
    for _, row in halo_subhalos.iterrows():
        sub_id_z0 = int(row['sub_id_z0'])
        tree = il.sublink.loadTree(myBasePath, 99, sub_id_z0, fields=TREE_FIELDS, onlyMPB=True)
        if tree is None:
            continue
        aux_array = np.zeros((tree['count'], 27))
        aux_array[:, 0] = sub_id_z0
        aux_array[:, 1] = tree['SubhaloGrNr']
        aux_array[:, 2] = tree['SubhaloPos'][:, 0] / 1e3  # cMpc/h
        aux_array[:, 3] = tree['SubhaloPos'][:, 1] / 1e3  # cMpc/h
        aux_array[:, 4] = tree['SubhaloPos'][:, 2] / 1e3  # cMpc/h
        aux_array[:, 5] = tree['SnapNum']
        aux_array[:, 6] = tree['SubhaloHalfmassRad'] / 1e3  # cMpc/h
        aux_array[:, 7] = tree['SubhaloIDRaw']
        aux_array[:, 8] = tree['SubhaloSFR']
        aux_array[:, 9] = tree['SubhaloMass']
        aux_array[:, 10:18] = tree['SubhaloStellarPhotometrics']
        aux_array[:, 18:24] = tree['SubhaloMassType']
        aux_array[:, 24:27] = tree['SubhaloVel']  # km/s
        filas.append(aux_array)
        del tree
        if len(filas) == nsubhalos_max:
            break
    if not filas:
        return None
    # [MEM] un solo concatenate en vez de np.vstack en cada iteración
    return np.concatenate(filas, axis=0)

In [20]:
def process_halo(hdf5_file, halo_id_z0, halo_subhalos, plotear=False, snap_min=60, nsubhalos_max=250,
                 vel_relativa_al_halo=True):
    """
    Procesa un halo completo y lo guarda en hdf5_file al final.

    vel_relativa_al_halo: si True, la dirección de movimiento del subhalo es v_sub - v_halo
    (movimiento respecto del cúmulo / ICM, lo relevante para ram pressure). Si False, se usa
    SubhaloVel tal cual (velocidad peculiar en el marco de la caja).
    """
    log_msg(f'-> Analizando Halo {halo_id_z0}  (RAM {mem_gb():.2f} GB)', to_screen=True)

    id_central_galaxy_z0 = int(halo_subhalos.iloc[0]['halo_id_central_galaxy_z0'])
    halo_evolution = load_halo_evolution(myBasePath, halo_id_z0, id_central_galaxy_z0)
    if halo_evolution is None:
        log_msg(f'   Halo {halo_id_z0}: sin merger tree de la central, se saltea', to_screen=True)
        return

    subhalos_list = cargar_subhalos_list(halo_subhalos, nsubhalos_max)
    if subhalos_list is None:
        log_msg(f'   Halo {halo_id_z0}: ningún subhalo con merger tree, se saltea', to_screen=True)
        return

    n = len(subhalos_list)
    subhalo_gas_num = np.full((n, 3), -99.)
    halo_gas_num = np.full((n, 3), -99.)
    subhalo_gas_mass = np.full((n, 3), -99.)
    halo_gas_mass = np.full((n, 3), -99.)
    subhalo_gas_intEnergy = np.full((n, 3), -99.)
    halo_gas_intEnergy = np.full((n, 3), -99.)
    subhalo_gas_elecAbundance = np.full((n, 3), -99.)
    halo_gas_elecAbundance = np.full((n, 3), -99.)
    subhalo_gas_T = np.full((n, 3), -99.)
    halo_gas_T = np.full((n, 3), -99.)
    subhalo_mass_bound = np.full(n, -99.)
    subhalo_mass_strip = np.full(n, -99.)
    subhalo_pos = np.full((n, 3), -99.)
    # Gas del halo entre 2 y 3 hmr en la dirección de la velocidad del subhalo
    halo_gas_num_vel = np.full(n, -99.)
    halo_gas_mass_vel = np.full(n, -99.)
    halo_gas_intEnergy_vel = np.full(n, -99.)
    halo_gas_elecAbundance_vel = np.full(n, -99.)
    halo_gas_T_vel = np.full(n, -99.)
    subhalo_vel_dir = np.full((n, 3), -99.)  # versor de la dirección usada

    def guardar(dst_num, dst_mass, dst_ie, dst_xe, dst_T, i, st):
        dst_num[i] = st['num'][:3]; dst_mass[i] = st['mass'][:3]; dst_ie[i] = st['intEnergy'][:3]
        dst_xe[i] = st['elecAbundance'][:3]; dst_T[i] = st['T'][:3]

    for snap in halo_evolution['snapshots']:
        if snap <= snap_min:
            continue
        t0 = time.time()
        halo_idx = np.where(halo_evolution['snapshots'] == snap)[0][0]
        haloID = halo_evolution['haloID'][halo_idx]
        if haloID is None or (isinstance(haloID, float) and np.isnan(haloID)):
            log_msg(f'   Halo {halo_id_z0} snap {snap}: sin haloID, se saltea', to_screen=True)
            continue
        halor200 = halo_evolution['r200'][halo_idx]
        halo_pos = np.array([halo_evolution['pos_x'][halo_idx],
                             halo_evolution['pos_y'][halo_idx],
                             halo_evolution['pos_z'][halo_idx]])
        halo_vel = halo_evolution['vel'][halo_idx]  # km/s

        xmin_plot, xmax_plot = halo_pos[0] - 5*halor200, halo_pos[0] + 5*halor200
        ymin_plot, ymax_plot = halo_pos[1] - 5*halor200, halo_pos[1] + 5*halor200

        # Todas las partículas de gas del halo (lo más pesado de todo el notebook)
        gas_halo = il.snapshot.loadHalo(myBasePath, snap, int(haloID), 'gas', fields=HALO_GAS_FIELDS)
        hay_gas_halo = gas_halo['count'] > 0
        if hay_gas_halo:
            # [MEM] coordenadas a cMpc/h IN PLACE (no se crea otra copia)
            halo_coords = gas_halo['Coordinates']
            halo_coords /= 1e3
            # [MEM] KD-tree una vez por snapshot: por subhalo solo se tocan las partículas cercanas
            halo_tree = cKDTree(halo_coords)
            n_halo_gas = len(halo_coords)

        ind_snap = np.where(subhalos_list[:, 5] == snap)[0]  # subhalos presentes en este snapshot
        for i in ind_snap:
            sub_id_z0 = int(subhalos_list[i, 0])
            hmr = subhalos_list[i, 6]
            log_msg(f'-> Analizando Halo {halo_id_z0} -> Snapshot {snap} -> subhalo {sub_id_z0}')

            if plotear:
                fig, ax = plt.subplots(1, 1)
                ax.set_title('Halo:' + str(halo_id_z0) + ' ; Subhalo:' + str(sub_id_z0) + ' ; snap:' + str(snap))
                path = 'graphs/Halo_' + str(halo_id_z0) + '/Subhalo_' + str(sub_id_z0)
                os.makedirs(path, exist_ok=True)
                ax.set_xlim(xmin_plot, xmax_plot)
                ax.set_ylim(ymin_plot, ymax_plot)

            sub_pos = subhalos_list[i, 2:5].copy()
            subhalo_pos[i, :] = sub_pos

            # Dirección de movimiento del subhalo (solo importa la dirección, no el módulo)
            v = subhalos_list[i, 24:27] - (halo_vel if vel_relativa_al_halo else 0.)
            v_norm = np.linalg.norm(v)
            vel_dir = v / v_norm if v_norm > 0 else None
            if vel_dir is not None:
                subhalo_vel_dir[i] = vel_dir

            # Gas del subhalo (subhaloIDRaw = subfindID + snapnum*1e12)
            sub_find_id = int(subhalos_list[i, 7] - snap*1e12)
            try:
                gas_subhalo = il.snapshot.loadSubhalo(myBasePath, snap, sub_find_id, 'gas', fields=SUB_GAS_FIELDS)
            except Exception:
                gas_subhalo = il.snapshot.loadSubhalo(myBasePath, snap, sub_find_id, 'gas', fields=SUB_GAS_FIELDS_SIN_POT)

            # ---------------- gas del halo alrededor del subhalo ----------------
            if hay_gas_halo:
                r_busqueda = max(3*hmr, 50/1000) * (1 + 1e-6) + 1e-9
                cand = np.asarray(halo_tree.query_ball_point(sub_pos, r_busqueda), dtype=np.int64)
                st = gas_stats(halo_coords[cand], gas_halo['Masses'][cand], gas_halo['InternalEnergy'][cand],
                               gas_halo['ElectronAbundance'][cand], sub_pos, halo_pos, hmr, vel_dir=vel_dir)
                guardar(halo_gas_num, halo_gas_mass, halo_gas_intEnergy, halo_gas_elecAbundance, halo_gas_T, i, st)
                if vel_dir is not None:
                    halo_gas_num_vel[i] = st['num'][3]
                    halo_gas_mass_vel[i] = st['mass'][3]
                    halo_gas_intEnergy_vel[i] = st['intEnergy'][3]
                    halo_gas_elecAbundance_vel[i] = st['elecAbundance'][3]
                    halo_gas_T_vel[i] = st['T'][3]

                if plotear:
                    rand_ind = np.random.randint(0, n_halo_gas, 10000)
                    ind_fix = cand[st['ind_fix']]
                    ax.scatter(halo_coords[rand_ind, 0], halo_coords[rand_ind, 1], color='black', s=5, zorder=0)
                    ax.scatter(halo_coords[ind_fix, 0], halo_coords[ind_fix, 1], color='darkcyan', s=2, marker='*', zorder=1)
                    ax.add_patch(plt.Circle((halo_pos[0], halo_pos[1]), halo_evolution['r200'][halo_idx], color='darkcyan', fill=False))
                    if vel_dir is not None:
                        ind_vel = cand[st['ind_vel']]
                        ax.scatter(halo_coords[ind_vel, 0], halo_coords[ind_vel, 1], color='gold', s=2, marker='o', zorder=4)
                        # flecha con la dirección de la velocidad (proyectada en x-y), largo 3 hmr
                        ax.annotate('', xy=(sub_pos[0] + 3*hmr*vel_dir[0], sub_pos[1] + 3*hmr*vel_dir[1]),
                                    xytext=(sub_pos[0], sub_pos[1]),
                                    arrowprops=dict(arrowstyle='->', color='gold', lw=1.5), zorder=5)
                del cand, st
            else:
                log_msg(f'-> Analizando Halo {halo_id_z0} -> Snapshot {snap} -> subhalo {sub_id_z0}: NO HALO GAS!!!')
                for arr in (halo_gas_num, halo_gas_mass, halo_gas_intEnergy, halo_gas_elecAbundance, halo_gas_T,
                            halo_gas_num_vel, halo_gas_mass_vel, halo_gas_intEnergy_vel,
                            halo_gas_elecAbundance_vel, halo_gas_T_vel):
                    arr[i] = 0
                if plotear:
                    ax.add_patch(plt.Circle((halo_pos[0], halo_pos[1]), halo_evolution['r200'][halo_idx], color='red', fill=False))

            # ---------------- gas propio del subhalo ----------------
            if gas_subhalo['count'] > 0:
                if 'Potential' in gas_subhalo:
                    vel2 = np.einsum('ij,ij->i', gas_subhalo['Velocities'], gas_subhalo['Velocities'])
                    energy = (gas_subhalo['Masses'] * gas_subhalo['Potential']
                              + gas_subhalo['Masses'] * gas_subhalo['InternalEnergy']
                              + 0.5 * gas_subhalo['Masses'] * vel2)
                    subhalo_mass_bound[i] = np.sum(gas_subhalo['Masses'][energy < 0])
                    subhalo_mass_strip[i] = np.sum(gas_subhalo['Masses'][energy > 0])
                    del vel2, energy

                sub_coords = gas_subhalo['Coordinates'] / 1e3  # cMpc/h
                st = gas_stats(sub_coords, gas_subhalo['Masses'], gas_subhalo['InternalEnergy'],
                               gas_subhalo['ElectronAbundance'], sub_pos, halo_pos, hmr)
                guardar(subhalo_gas_num, subhalo_gas_mass, subhalo_gas_intEnergy, subhalo_gas_elecAbundance, subhalo_gas_T, i, st)

                if plotear:
                    rand_ind = np.random.randint(0, len(sub_coords), 10000)
                    ax.scatter(sub_coords[rand_ind, 0], sub_coords[rand_ind, 1], color='coral', s=4, marker='+', zorder=2)
                    ax.scatter(sub_coords[st['ind_fix'], 0], sub_coords[st['ind_fix'], 1], color='coral', s=2, marker='*', zorder=3)
                    ax.add_patch(plt.Circle((sub_pos[0], sub_pos[1]), 2*hmr, color='coral', fill=False, linestyle='--'))
                del sub_coords, st
            else:
                log_msg(f'-> Analizando Halo {halo_id_z0} -> Snapshot {snap} -> subhalo {sub_id_z0}: NO SUBHALO GAS!!!')
                subhalo_mass_bound[i] = 0
                subhalo_mass_strip[i] = 0
                for arr in (subhalo_gas_num, subhalo_gas_mass, subhalo_gas_intEnergy, subhalo_gas_elecAbundance, subhalo_gas_T):
                    arr[i] = 0
                if plotear:
                    ax.add_patch(plt.Circle((sub_pos[0], sub_pos[1]), 2*hmr, color='red', fill=False, linestyle='--'))

            if plotear:
                fig.savefig(path + '/Subhalo_' + str(sub_id_z0) + '_snap_' + str(snap) + '.png', dpi=150, bbox_inches='tight')
                plt.close(fig)
            del gas_subhalo

        # [MEM] liberar el gas del cúmulo ANTES de cargar el del snapshot siguiente
        if hay_gas_halo:
            del halo_tree, halo_coords
        del gas_halo
        plt.close('all')
        liberar_memoria()
        log_msg(f'   Halo {halo_id_z0} snap {snap}: {len(ind_snap)} subhalos en {time.time()-t0:.0f} s, RAM {mem_gb():.2f} GB',
                to_screen=True)

    # [MEM] Se escribe todo al final: si el kernel muere a mitad del halo no queda un grupo incompleto
    log_msg('Guardando datos ...', to_screen=True)
    halo_group = hdf5_file.create_group(f'halo_{halo_id_z0}')
    halo_group.attrs['vel_relativa_al_halo'] = vel_relativa_al_halo
    halo_group.create_dataset('vel', data=halo_evolution['vel'])  # km/s
    for key in ('pos_x', 'pos_y', 'pos_z', 'r200', 'm200', 'snapshots'):
        halo_group.create_dataset(key, data=halo_evolution[key])

    for sub_ind in np.unique(subhalos_list[:, 0]):
        aux_ind = np.where(subhalos_list[:, 0] == sub_ind)[0]
        g = halo_group.create_group(f'subhalo_{int(sub_ind)}')
        g.create_dataset('Subhalo_pos', data=subhalo_pos[aux_ind])
        g.create_dataset('snapshots', data=subhalos_list[aux_ind, 5])  # nuevo: para alinear por snapshot
        g.create_dataset('pos_x', data=subhalos_list[aux_ind, 2])      # cMpc/h
        g.create_dataset('pos_y', data=subhalos_list[aux_ind, 3])      # cMpc/h
        g.create_dataset('pos_z', data=subhalos_list[aux_ind, 4])      # cMpc/h
        g.create_dataset('halfMassRad', data=subhalos_list[aux_ind, 6])
        g.create_dataset('SubhaloSFR', data=subhalos_list[aux_ind, 8])
        g.create_dataset('SubhaloMass', data=subhalos_list[aux_ind, 9])
        g.create_dataset('SubhaloMags', data=subhalos_list[aux_ind, 10:18])
        g.create_dataset('SubhaloMassType', data=subhalos_list[aux_ind, 18:24])
        g.create_dataset('Subhalo_gas_mass', data=subhalo_gas_mass[aux_ind])
        g.create_dataset('Subhalo_gas_num', data=subhalo_gas_num[aux_ind])
        g.create_dataset('Subhalo_gas_T', data=subhalo_gas_T[aux_ind])
        g.create_dataset('Halo_gas_mass', data=halo_gas_mass[aux_ind])
        g.create_dataset('Halo_gas_num', data=halo_gas_num[aux_ind])
        g.create_dataset('Halo_gas_T', data=halo_gas_T[aux_ind])
        g.create_dataset('Subhalo_mass_bound', data=subhalo_mass_bound[aux_ind])
        g.create_dataset('Subhalo_mass_strip', data=subhalo_mass_strip[aux_ind])
        # Gas del halo entre 2 y 3 hmr en la dirección de la velocidad del subhalo
        g.create_dataset('Subhalo_vel', data=subhalos_list[aux_ind, 24:27])    # km/s
        g.create_dataset('Subhalo_vel_dir', data=subhalo_vel_dir[aux_ind])      # versor usado
        g.create_dataset('Halo_gas_num_vel', data=halo_gas_num_vel[aux_ind])
        g.create_dataset('Halo_gas_mass_vel', data=halo_gas_mass_vel[aux_ind])
        g.create_dataset('Halo_gas_T_vel', data=halo_gas_T_vel[aux_ind])
    hdf5_file.flush()

In [28]:
hdf5_file = h5py.File('data_tree_2.hdf5', 'r')

In [29]:
hdf5_file.keys()

<KeysViewHDF5 ['halo_2']>

In [ ]:
hdf5_file.close()

In [27]:
%%time

# --------------------------------------------------------------------------------------------
# Se tarda ~ 2:40 hs en analizar ~480 subhalos del halo 2 hasta el snapshot 60 (z=0.68, lookbacktime = 6.3 Gyr)
# El detalle por subhalo se escribe en log_download.txt (en pantalla solo un resumen por snapshot)
# --------------------------------------------------------------------------------------------

plotear = True
nmax_halos = 1       # Numero de halos a analizar
snap_min = 60        # Snapshot minimo a analizar
nsubhalos_max = 2000 # Numero de subhalos a analizar por halo
max_time = 24        # hs
vel_relativa_al_halo = True  # dirección de v_sub - v_halo (False: SubhaloVel tal cual)

init_hour = time.time()
with h5py.File('data_tree_2.hdf5', 'a') as hdf5_file:
    analized_halos = set(hdf5_file.keys())
    contador_halos = 0

    for halo_id_z0, halo_subhalos in tqdm(data.groupby('halo_id_z0'), desc='Halos'):
        if ((time.time() - init_hour) / 3600) > max_time:
            break
        if contador_halos >= nmax_halos:
            break

        halo_id_z0 = int(halo_id_z0)
        if (halo_id_z0 <= 1) or (f'halo_{halo_id_z0}' in analized_halos):
            continue

        contador_halos += 1
        process_halo(hdf5_file, halo_id_z0, halo_subhalos,
                     plotear=plotear, snap_min=snap_min, nsubhalos_max=nsubhalos_max,
                     vel_relativa_al_halo=vel_relativa_al_halo)

        # [MEM] al volver de la función ya no queda ninguna variable del halo viva
        liberar_memoria()
        print(f'RAM despues del halo {halo_id_z0}: {mem_gb():.2f} GB')

Halos:   0%|          | 0/549 [00:00<?, ?it/s]

-> Analizando Halo 2  (RAM 0.23 GB)
Warning, empty return. Subhalo [18782] at snapNum [99] not in tree.
   Halo 2 snap 99: 481 subhalos en 239 s, RAM 0.25 GB
   Halo 2 snap 98: 480 subhalos en 226 s, RAM 0.25 GB
   Halo 2 snap 97: 481 subhalos en 232 s, RAM 0.25 GB
   Halo 2 snap 96: 481 subhalos en 234 s, RAM 0.25 GB
   Halo 2 snap 95: 480 subhalos en 227 s, RAM 0.25 GB
   Halo 2 snap 94: 481 subhalos en 229 s, RAM 0.25 GB
   Halo 2 snap 93: 481 subhalos en 232 s, RAM 0.25 GB
   Halo 2 snap 92: 481 subhalos en 238 s, RAM 0.25 GB
   Halo 2 snap 91: 480 subhalos en 231 s, RAM 0.25 GB
   Halo 2 snap 90: 481 subhalos en 238 s, RAM 0.25 GB
   Halo 2 snap 89: 480 subhalos en 228 s, RAM 0.25 GB
   Halo 2 snap 88: 481 subhalos en 229 s, RAM 0.25 GB
   Halo 2 snap 87: 481 subhalos en 238 s, RAM 0.25 GB
   Halo 2 snap 86: 481 subhalos en 235 s, RAM 0.25 GB
   Halo 2 snap 85: 481 subhalos en 230 s, RAM 0.25 GB
   Halo 2 snap 84: 481 subhalos en 241 s, RAM 0.25 GB
   Halo 2 snap 83: 481 subhalos 

Halos:   0%|          | 2/549 [2:35:36<709:20:26, 4668.42s/it]

RAM despues del halo 2: 0.24 GB
CPU times: user 2h 16min 21s, sys: 22min 8s, total: 2h 38min 30s
Wall time: 2h 35min 36s


# PASO 5: ELIMINAR SUBHALOS SIN MERGER TREE

In [ ]:
valid_sub_ids = []

with h5py.File('data_tree.hdf5', "r") as f:
    for halo_name in f.keys():
        halo_group = f[halo_name]

        # Buscar subhalos
        subhalos_grp = [k for k in halo_group.keys() if k.startswith("subhalo_")]

        # Extraer el número de cada subhalo y guardarlo
        for sub_name in subhalos_grp:
            sub_id = int(sub_name.split("_")[1])
            if sub_id == 1:
                print(halo_name)
            valid_sub_ids.append(sub_id)

# Convertimos a set para buscar rápido
valid_sub_ids = set(valid_sub_ids)

print(f"Total subhalos en merger tree: {len(valid_sub_ids)}")
print(f"Primeros 20 ids: {list(valid_sub_ids)[:20]}")

In [ ]:
data_clean = data[data["sub_id_z0"].isin(valid_sub_ids)].copy()

print(f"Subhalos originales en DataFrame: {len(data)}")
print(f"Subhalos después de limpieza: {len(data_clean)}")

In [ ]:
# Eliminamos duplicados
data_clean_final = data_clean.drop_duplicates(subset=['sub_id_z0'])
print(len(data_clean_final))

In [ ]:
data_clean_final.to_csv('link_halo_subhalos_z0.dat', sep=' ', index=False)

In [ ]:
data_clean_final.head()